# catnip local — stage 1 training on the dev machine (M-series, MPS)

**Not the production training path.** Full runs belong on a spot GPU
(`scripts/train/spot_bootstrap.sh`, runbook §B3) — MPS is roughly 3-5x
slower than even a T4, so 150 epochs locally takes a week-plus. Use this
notebook for short test runs, resume/debug of checkpoints, or the final
model pull-back from R2.

Data: prefer the mounted HDD copy (`/Volumes/rifuSSD/catnip-data`); if it
is not mounted, pull the 14 GB sliced tarball from R2 to `~/catnip-data`.
Config: `config/pipeline.yaml` (device=mps, epochs=150, patience=15).


### 0. R2 credentials


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

CONF = Path.home() / "secrets" / "r2-sippy.conf"
if not (os.environ.get("R2_ACCESS_KEY_ID") and os.environ.get("R2_SECRET_ACCESS_KEY")):
    if not CONF.exists():
        raise FileNotFoundError(
            f"No R2 credentials: export R2_ACCESS_KEY_ID/R2_SECRET_ACCESS_KEY "
            f"or create {CONF} (KEY=VALUE lines)."
        )
    for line in CONF.read_text().splitlines():
        if "=" in line and not line.startswith("#"):
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip())

if not (os.environ["R2_ACCESS_KEY_ID"] and os.environ["R2_SECRET_ACCESS_KEY"]):
    raise RuntimeError("R2 credentials incomplete (empty key or secret).")

R2_ENDPOINT = "https://eccb0b608bd926e793224944aba32cf4.r2.cloudflarestorage.com"
BUCKET = "catnip-canonical"

try:
    import boto3
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "boto3"], check=True)
    import boto3

import os as _os
s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=_os.environ["R2_ACCESS_KEY_ID"],
    aws_secret_access_key=_os.environ["R2_SECRET_ACCESS_KEY"],
    region_name="auto",
)
print("R2 client ready:", R2_ENDPOINT)


### 1. Data source — HDD if mounted, else R2 pull


In [1]:
import os, shutil, tarfile
from pathlib import Path

HDD_ROOT = Path("/Volumes/rifuSSD/catnip-data")
LOCAL_ROOT = Path.home() / "catnip-data"
TAR_KEY = "compressed/stage1_sliced.tar"

if HDD_ROOT.exists():
    CATNIP_DATA = HDD_ROOT
    print("Using HDD copy:", CATNIP_DATA)
else:
    CATNIP_DATA = LOCAL_ROOT
    sliced_yaml = CATNIP_DATA / "training" / "stage1_sliced" / "dataset.yaml"
    if sliced_yaml.exists():
        print("Using existing local copy:", CATNIP_DATA)
    else:
        print(f"HDD not mounted; pulling {TAR_KEY} from R2 (~14 GB)...")
        CATNIP_DATA.mkdir(parents=True, exist_ok=True)
        tar_path = CATNIP_DATA / "stage1_sliced.tar"
        s3.download_file(BUCKET, TAR_KEY, str(tar_path))
        with tarfile.open(tar_path) as tf:
            tf.extractall(CATNIP_DATA)   # extractall: trusted internal artifact
        tar_path.unlink()
        print("Extracted to", CATNIP_DATA)

os.environ["CATNIP_DATA"] = str(CATNIP_DATA)

# dataset.yaml ships with the generating host's absolute path — repoint it.
ds = CATNIP_DATA / "training" / "stage1_sliced" / "dataset.yaml"
if not ds.exists():
    raise FileNotFoundError(f"dataset.yaml missing at {ds} — data source is incomplete.")
lines = [f"path: {CATNIP_DATA / 'training' / 'stage1_sliced'}\n" if l.startswith("path:")
         else l for l in ds.read_text().splitlines(keepends=True)]
ds.write_text("".join(lines))
print("dataset.yaml path ->", CATNIP_DATA / "training" / "stage1_sliced")


Using HDD copy: /Volumes/rifuSSD/catnip-data
dataset.yaml path -> /Volumes/rifuSSD/catnip-data/training/stage1_sliced


### 2. Train (pixi env, device=mps)

Full config run: epochs=150, patience=15 (config defaults). For a quick
smoke test set `EPOCHS` below to something small (e.g. 3) before running.


In [2]:
import shutil
import subprocess
from pathlib import Path

EPOCHS = None   # None = use config (150); set an int (e.g. 3) for smoke tests
EXTRA = ["--override", "training.stage1.device=mps"]
if EPOCHS is not None:
    EXTRA += ["--override", f"training.stage1.epochs={EPOCHS}"]

# VS Code/Jupyter set cwd to the notebook's own dir — walk up to the
# repo root (nearest ancestor holding config/pipeline.yaml).
REPO_DIR = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config" / "pipeline.yaml").exists()),
    None,
)
if REPO_DIR is None:
    raise FileNotFoundError(f"No config/pipeline.yaml at or above cwd={Path.cwd()}.")
if shutil.which("pixi") is None:
    raise RuntimeError("pixi not on PATH — install: curl -fsSL https://pixi.sh/install.sh | sh")

cmd = ["pixi", "run", "train-stage1"] + EXTRA
print("$", " ".join(cmd), "\n")
result = subprocess.run(cmd, cwd=REPO_DIR)
if result.returncode != 0:
    raise RuntimeError(f"train-stage1 exited with {result.returncode}")
print("train-stage1 finished cleanly")


$ pixi run train-stage1 --override training.stage1.device=mps 



 WARN Encountered 1 warning while parsing the manifest:
  ⚠ the `[system-requirements]` table is deprecated in favor of virtual
  │ packages on `platforms`
    ╭─[/Users/rifusaki/localRepos/personal/catnip/pixi.toml:38:1]
 37 │     # Build-string pin selects the conda-forge CUDA variant (default is cpu_mkl).
 38 │ ╭─▶ [feature.cuda.system-requirements]
 39 │ ├─▶ cuda = "12.6"
    · ╰──── declare these on the `platforms` entries instead
 40 │     
    ╰────
  help: e.g. platforms = [{ name = "my-linux-64", platform = "linux-64", cuda
        = "12.6" }]

✨ Pixi task (train-stage1 in default): python scripts/train/stage1.py --override training.stage1.device=mps
INFO     __main__             Starting Stage 1 training: model=/Volumes/rifuSSD/catnip-data/models/yolo26n.pt, device=mps, epochs=150, batch=8


New https://pypi.org/project/ultralytics/8.4.194 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.193 🚀 Python-3.11.14 torch-2.4.1 MPS (Apple M3)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/Volumes/rifuSSD/catnip-data/training/stage1_sliced/dataset.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/Vo

Traceback (most recent call last):
  File "/Users/rifusaki/localRepos/personal/catnip/scripts/train/stage1.py", line 189, in <module>
    main()
  File "/Users/rifusaki/localRepos/personal/catnip/scripts/train/stage1.py", line 152, in main
    model.train(**train_kwargs)
  File "/Users/rifusaki/localRepos/personal/catnip/.pixi/envs/default/lib/python3.11/site-packages/ultralytics/engine/model.py", line 901, in train
    self.trainer.train()
  File "/Users/rifusaki/localRepos/personal/catnip/.pixi/envs/default/lib/python3.11/site-packages/ultralytics/engine/trainer.py", line 263, in train
    self._do_train()
  File "/Users/rifusaki/localRepos/personal/catnip/.pixi/envs/default/lib/python3.11/site-packages/ultralytics/engine/trainer.py", line 622, in _do_train
    pbar.set_description(
  File "/Users/rifusaki/localRepos/personal/catnip/.pixi/envs/default/lib/python3.11/site-packages/ultralytics/utils/tqdm.py", line 322, in set_description
    def set_description(self, desc: str | None) 

KeyboardInterrupt: 

### 3. Verify


In [ ]:
import os, sys
from pathlib import Path

REPO_DIR = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config" / "pipeline.yaml").exists()),
    None,
)
if REPO_DIR is None:
    raise FileNotFoundError(f"No config/pipeline.yaml at or above cwd={Path.cwd()}.")
sys.path.insert(0, str(REPO_DIR))

from src.config import load_settings

settings = load_settings(str(REPO_DIR / "config" / "pipeline.yaml"))
best = settings.paths.model_dir / "yolo26_stage1_body_head_face.pt"

if not best.exists():
    raise FileNotFoundError(f"Deliverable missing: {best}")
size_mb = best.stat().st_size / 1e6
if size_mb < 1:
    raise RuntimeError(f"Deliverable suspiciously small: {size_mb:.2f} MB")
print(f"OK: {best} ({size_mb:.1f} MB)")


### 4. Push results to R2


In [ ]:
import tarfile, tempfile
from pathlib import Path

MODEL_KEY = "models/yolo26_stage1_body_head_face.pt"
model = settings.paths.model_dir / "yolo26_stage1_body_head_face.pt"

s3.upload_file(str(model), BUCKET, MODEL_KEY)
print("pushed:", MODEL_KEY)

run_dir = settings.paths.runs_dir / "detect" / "stage1"
if run_dir.exists():
    with tempfile.NamedTemporaryFile(suffix=".tar", delete=False) as tf:
        run_tar = Path(tf.name)
    with tarfile.open(run_tar, "w") as tar:
        tar.add(run_dir, arcname="runs/detect/stage1")
    s3.upload_file(str(run_tar), BUCKET, "runs/stage1-run.tar")
    run_tar.unlink()
    print("pushed: runs/stage1-run.tar")
else:
    print("no run dir to push — skipping")
print("Done.")
